# Same-boot GPU benchmark of `baked_runtime`, M1_210210

- `scripts/benchmark_baked_runtime.py` on one RTX 5060 Ti; arms interleaved inside each round.
- `free_current` is `baked_runtime=False`; `free_baked` is `True`; `fixed` has $z$ and $\sigma_\star$ fixed.

In [1]:
import json
from pathlib import Path
import pandas as pd

OUT = Path.cwd() if Path.cwd().name == "runtime-z-sigma-speed" else Path.cwd() / "results/runtime-z-sigma-speed"
gpu = json.loads((OUT / "timing-gpu.json").read_text())
rental = json.loads(next(OUT.glob("vast_run_*.json")).read_text())
print(gpu["device"], "| jax", gpu["jax"], "| x64", gpu["x64"], "|", gpu["rounds_per_arm"], "rounds x", gpu["repeats_per_round"], "calls")
print(rental["offer"]["gpu_name"], f"${rental['offer']['dph_total']:.4f}/h", "| spent $", rental["spent_usd"], "| instances left", rental["instances_left"])
pd.DataFrame(gpu["summary"]).set_index("particles").round(3)

cuda:0 | jax 0.10.2 | x64 True | 30 rounds x 20 calls
RTX 5060 Ti $0.1269/h | spent $ 0.0315 | instances left []


,fixed_us_per_call,free_current_us_per_call,free_baked_us_per_call,speedup_median,speedup_min,speedup_max
particles,,,,,,
100,18.511,45.409,31.325,1.45,1.445,1.456
500,16.299,45.875,29.978,1.53,1.528,1.535


- Spread of the per-round speed-up, and the log-likelihood difference on the prior draws.

In [2]:
rounds = pd.DataFrame(gpu["rounds"])
display(rounds.groupby("particles")[["fixed", "free_current", "free_baked", "speedup"]].agg(["min", "median", "max"]).round(3))
pd.Series({k: f"{v:.3g}" if isinstance(v, float) else v for k, v in gpu["log_likelihood"].items()})

fixed                 free_current                 free_baked  \
              min  median     max          min  median     max        min   
particles                                                                   
100        18.151  18.511  18.571       45.159  45.409  45.510     31.045   
500        16.130  16.299  16.319       45.711  45.875  46.012     29.809   

                          speedup                
           median     max     min median    max  
particles                                        
100        31.325  31.418   1.445   1.45  1.456  
500        29.978  30.013   1.528   1.53  1.535

draws                          500
finite                        True
lnl_min                  -1.33e+11
lnl_max                   2.21e+05
max_abs_dlnl              3.73e-09
max_rel_dlnl              1.02e-13
within_test_tolerance         True
dtype: object